In [8]:
import boto3
import json
import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

def invoke_model(bedrock, model_id, text, request_id):
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 10,
        "messages": [{"role": "user", "content": text}]
    }
    try:
        start = time.time()
        response = bedrock.invoke_model(modelId=model_id, body=json.dumps(body))
        latency = time.time() - start
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {request_id}: SUCCESS ({latency:.2f}s)")
        return (request_id, "SUCCESS")
    except Exception as e:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {request_id}: ERROR - {str(e)}")
        return (request_id, "ERROR")

def test_rpm_quotas(throttle_model, throttle_name, other_model, other_name, region='us-east-1'):
    bedrock = boto3.client('bedrock-runtime', region_name=region)
    text = "test"
    
    throttle_results = []
    other_results = []
    
    print(f"\nSending 1000 requests to {throttle_name} and 150 to {other_name} over 60 seconds")
    print("="*80)
    
    start_time = time.time()
    end_time = start_time + 60
    
    with ThreadPoolExecutor(max_workers=100) as executor:
        futures = []
        throttle_count = 0
        other_count = 0
        
        while time.time() < end_time:
            for i in range(50):
                futures.append(executor.submit(
                    invoke_model, bedrock, throttle_model, text, f"{throttle_name}-{throttle_count}"
                ))
                throttle_count += 1
            
            for i in range(7):
                futures.append(executor.submit(
                    invoke_model, bedrock, other_model, text, f"{other_name}-{other_count}"
                ))
                other_count += 1
            
            time.sleep(3)
        
        for future in as_completed(futures):
            request_id, status = future.result()
            if throttle_name in request_id:
                throttle_results.append(status)
            else:
                other_results.append(status)
    
    print("\n" + "="*80)
    print("RESULTS")
    print("="*80)
    
    throttle_success = throttle_results.count("SUCCESS")
    throttle_error = throttle_results.count("ERROR")
    throttle_total = len(throttle_results)
    
    other_success = other_results.count("SUCCESS")
    other_error = other_results.count("ERROR")
    other_total = len(other_results)
    
    print(f"\n{throttle_name}:")
    print(f"  Total: {throttle_total}")
    print(f"  Success: {throttle_success} ({throttle_success/throttle_total*100:.1f}%)")
    print(f"  Error: {throttle_error} ({throttle_error/throttle_total*100:.1f}%)")
    
    print(f"\n{other_name}:")
    print(f"  Total: {other_total}")
    print(f"  Success: {other_success} ({other_success/other_total*100:.1f}%)")
    print(f"  Error: {other_error} ({other_error/other_total*100:.1f}%)")

US_MODEL = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
GLOBAL_MODEL = "global.anthropic.claude-sonnet-4-5-20250929-v1:0"

print("Which profile to throttle?")
print("1. US CRIS")
print("2. Global CRIS")
choice = input("Enter 1 or 2: ").strip()

if choice == "1":
    test_rpm_quotas(US_MODEL, "US-CRIS", GLOBAL_MODEL, "Global-CRIS")
else:
    test_rpm_quotas(GLOBAL_MODEL, "Global-CRIS", US_MODEL, "US-CRIS")

Which profile to throttle?
1. US CRIS
2. Global CRIS


Enter 1 or 2:  1



Sending 1000 requests to US-CRIS and 150 to Global-CRIS over 60 seconds
[01:46:13] US-CRIS-6: SUCCESS (1.61s)
[01:46:13] US-CRIS-9: SUCCESS (1.64s)
[01:46:13] US-CRIS-5: SUCCESS (1.70s)
[01:46:13] US-CRIS-22: SUCCESS (1.52s)
[01:46:13] US-CRIS-3: SUCCESS (1.77s)
[01:46:13] US-CRIS-7: SUCCESS (1.71s)
[01:46:13] US-CRIS-14: SUCCESS (1.69s)
[01:46:13] US-CRIS-0: SUCCESS (1.82s)
[01:46:13] US-CRIS-2: SUCCESS (1.82s)
[01:46:13] US-CRIS-1: SUCCESS (1.87s)
[01:46:13] US-CRIS-21: SUCCESS (1.65s)
[01:46:13] US-CRIS-11: SUCCESS (1.79s)
[01:46:13] US-CRIS-20: SUCCESS (1.69s)
[01:46:13] US-CRIS-13: SUCCESS (1.81s)
[01:46:14] US-CRIS-19: SUCCESS (1.75s)
[01:46:14] US-CRIS-44: SUCCESS (1.47s)
[01:46:14] US-CRIS-34: SUCCESS (1.59s)
[01:46:14] US-CRIS-30: SUCCESS (1.65s)
[01:46:14] US-CRIS-25: SUCCESS (1.72s)
[01:46:14] US-CRIS-15: SUCCESS (1.83s)
[01:46:14] US-CRIS-10: SUCCESS (1.89s)
[01:46:14] US-CRIS-8: SUCCESS (1.91s)
[01:46:14] US-CRIS-16: SUCCESS (1.87s)
[01:46:14] US-CRIS-27: SUCCESS (1.76s)
